# trainscope — multi-GPU validation (Kaggle, 2× T4)

Runs the protocol from `docs/VALIDATION.md` on **real NCCL/CUDA**, free, on
Kaggle's 2×T4 notebook tier.

**Before running:** Notebook settings (right sidebar) → *Accelerator* →
**GPU T4 ×2**. Then *Session options* → Internet **On** (needed to clone the
repo and `pip install`).

Cells are ordered to match Experiments 1–3 in `docs/VALIDATION.md`. Run them
top to bottom; each cell's output is the artifact to save back into
`docs/validation-runs/`.

## 0. Confirm 2 GPUs are visible

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("CUDA available:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, "Need 2 GPUs — set Accelerator to GPU T4 x2 in notebook settings"

## 1. Clone trainscope and install

In [ ]:
!rm -rf trainscope
!git clone --depth 1 https://github.com/Sumu004/trainscope.git
%cd trainscope
!pip install -q -e ".[torch]"
!python -c "import trainscope, torch; print('trainscope', trainscope.__version__, '| torch', torch.__version__)"

## 2. Experiment 1 — Straggler attribution (known answer: rank 1)

Real NCCL DDP across both GPUs; rank 1 does extra compute each step. The
multi-rank critical-path analyzer should name **rank 1** as a persistent
straggler.

**Acceptance:** `DIST.STRAGGLER` fires naming rank 1, with `slowest_fraction`
≫ 1/world_size, a positive z-score, and `wall_frac_lost_to_imbalance` in the
right ballpark for the injected slowdown.

In [ ]:
!rm -rf runs/gpu_straggler
!torchrun --standalone --nproc_per_node=2 examples/ddp_gloo.py \
    --steps 200 --straggler-rank 1 --run-dir runs/gpu_straggler

In [ ]:
!python -m trainscope.cli analyze runs/gpu_straggler | tee exp1_straggler.txt

## 3. Experiment 2 — Exposed communication (known answer: bad vs good overlap)

Two configs, same model: **(a)** tiny per-GPU batch (all-reduce can't hide
behind compute → high *exposed* comm) vs **(b)** large per-GPU batch (default
DDP bucketing hides it well → low exposed comm). Each captures a real
`torch.profiler` Kineto trace.

**Acceptance:** `overlap_efficiency(b) ≫ overlap_efficiency(a)`; `DIST.EXPOSED_COMM`
fires HIGH for (a) and is absent/LOW for (b).

In [ ]:
%%writefile exposed_comm_probe.py
"""Capture a real Kineto trace for a small DDP job at a given batch size.
Usage: torchrun --standalone --nproc_per_node=2 exposed_comm_probe.py --batch 8 --out runs/exp2_bad
"""
from __future__ import annotations
import argparse, os
import torch, torch.distributed as dist, torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.profiler import profile, ProfilerActivity

ap = argparse.ArgumentParser()
ap.add_argument("--batch", type=int, default=8)
ap.add_argument("--steps", type=int, default=20)
ap.add_argument("--out", default="runs/exposed_comm_probe")
args = ap.parse_args()

rank = int(os.environ["RANK"]); world_size = int(os.environ["WORLD_SIZE"])
dist.init_process_group("nccl", rank=rank, world_size=world_size)
device = torch.device(f"cuda:{rank}")
torch.cuda.set_device(device)
torch.manual_seed(0)

# A wide model (more grad to all-reduce) so under-batching exposes comm clearly.
model = nn.Sequential(nn.Linear(2048, 2048), nn.ReLU(), nn.Linear(2048, 2048)).to(device)
model = DDP(model, device_ids=[rank])
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

def step():
    x = torch.randn(args.batch, 2048, device=device)
    y = torch.randn(args.batch, 2048, device=device)
    loss = loss_fn(model(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()

for _ in range(5):
    step()  # warmup

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as p:
    for _ in range(args.steps):
        step()

if rank == 0:
    os.makedirs(args.out, exist_ok=True)
    p.export_chrome_trace(f"{args.out}/trace.json")
    print(f"wrote {args.out}/trace.json (batch={args.batch})")
dist.barrier(); dist.destroy_process_group()

In [ ]:
# (a) Bad overlap: tiny per-GPU batch.
!rm -rf runs/exp2_bad && torchrun --standalone --nproc_per_node=2 exposed_comm_probe.py --batch 4 --out runs/exp2_bad
!python -m trainscope.cli analyze runs/exp2_bad | tee exp2_bad_overlap.txt

In [ ]:
# (b) Good overlap: large per-GPU batch, same model, default bucketing.
!rm -rf runs/exp2_good && torchrun --standalone --nproc_per_node=2 exposed_comm_probe.py --batch 256 --out runs/exp2_good
!python -m trainscope.cli analyze runs/exp2_good | tee exp2_good_overlap.txt

## 4. Experiment 3 — MFU sanity vs a known model

Run `examples/efficiency_mfu.py` (or any `AutoProfiler(measure_flops=True)`
job) on a single T4 — FLOPs are auto-counted, peak comes from the hardware
table (T4 ≈ 65 TFLOP/s fp16). Cross-check the reported MFU against
`6 · N_params · tokens / (step_time · peak_FLOPS)`.

**Acceptance:** reported MFU is within ~15% of the hand-computed value, and
never exceeds 100%.

In [ ]:
!python examples/efficiency_mfu.py   # fixed-path demo; writes runs/mfu
!rm -rf runs/exp3_mfu && cp -r runs/mfu runs/exp3_mfu
!python -m trainscope.cli analyze runs/exp3_mfu --peak-tflops 65 | tee exp3_mfu.txt

## 5. Collect the artifacts

Zip everything trainscope produced plus the console captures — this is what
goes back into `docs/validation-runs/` in the repo (each `run.json` carries
the provenance: torch version, CUDA version, GPU name, world size, etc.).

In [ ]:
!zip -r gpu_validation_artifacts.zip \
    exp1_straggler.txt exp2_bad_overlap.txt exp2_good_overlap.txt exp3_mfu.txt \
    runs/gpu_straggler/*/run.json runs/exp2_bad/run.json runs/exp2_bad/trace.json \
    runs/exp2_good/run.json runs/exp2_good/trace.json runs/exp3_mfu/run.json
print("Download gpu_validation_artifacts.zip from the notebook's Output panel,")
print("then unzip it into docs/validation-runs/<date>-kaggle-2xT4/ in your trainscope checkout.")